# SymView — analysis notebook

This notebook is the *analysis* stage of the replication package.  It reads only
`results/raw/**` and the frozen configuration, and reproduces every table and
figure cited by the report.  Run the experiments first:

```bash
python experiments/run_experiments.py
python experiments/run_lab_models.py
```


In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 160)
print("project:", ROOT)


## 1. Frozen subset and environment

In [ ]:
import yaml
subset = yaml.safe_load((ROOT / "config" / "subset.yaml").read_text(encoding="utf-8"))
models = [m["model_id"] for m in subset["models"]]
env = json.loads((ROOT / "results" / "raw" / "environment.json").read_text(encoding="utf-8"))
print(json.dumps(env, indent=2, ensure_ascii=False))
print("models:", len(models), "families:", sorted({m["family"] for m in subset["models"]}))


## 2. Analysis verdicts (RQ3 / RQ5)

In [ ]:
rows = []
for path in sorted((ROOT / "results" / "raw" / "analysis").glob("*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    rows.append({
        "model": data["model"]["model_id"],
        "symmetry_set": ",".join(data["symmetry"]["selected"]) or "-",
        "view_drops": ",".join(data["history"]["dropped"]) or "-",
        "liveness": data["program"]["liveness"],
        "candidates": len(data["symmetry"]["candidates"]),
        "accepted": sum(1 for c in data["symmetry"]["candidates"] if c["verdict"] == "sound"),
    })
analysis = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
display(analysis)
print("symmetry coverage:", (analysis.symmetry_set != "-").mean())
print("view coverage   :", (analysis.view_drops != "-").mean())


## 3. Reduction per arm (RQ1 / RQ4)

In [ ]:
def arm_run(model, arm):
    path = ROOT / "results" / "raw" / "runs" / f"{model}__{arm}__r0.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))["run"]

rows = []
for model in models:
    plain = arm_run(model, "plain")
    if not plain or not plain.get("distinct_states"):
        continue
    row = {"model": model, "plain": plain["distinct_states"],
           "plain_s": plain.get("wall_seconds")}
    for arm in ("symmetry", "view", "both"):
        run = arm_run(model, arm)
        if run and run.get("distinct_states"):
            row[arm] = run["distinct_states"]
            row[arm + "_reduction"] = 1 - run["distinct_states"] / row["plain"]
    rows.append(row)
reduction = pd.DataFrame(rows).set_index("model")
display(reduction.style.format(precision=3))
print("median reduction (both):", reduction.get("both_reduction").median())


## 4. Violation preservation (RQ2)

In [ ]:
rows = []
for path in sorted((ROOT / "results" / "raw" / "mutants").glob("*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    for mutant in data["mutants"]:
        if not mutant.get("detected"):
            continue
        line = {"model": data["model_id"], "mutant": mutant["mutant_id"],
                "operator": mutant["operator"], "baseline": mutant["baseline_verdict"]}
        for arm in ("symmetry", "view", "both"):
            run = mutant["arms"].get(arm, {})
            line[arm] = run.get("verdict")
        rows.append(line)
mutants = pd.DataFrame(rows)
display(mutants if len(mutants) else "no mutant was detected by the baseline")
if len(mutants):
    for arm in ("symmetry", "view", "both"):
        print(arm, "preserved:", (mutants[arm] == "violation").mean())


## 5. Figures

Generate (or refresh) the figures used by the report:

In [ ]:
import subprocess
for script in ("make_tables.py", "make_figures.py"):
    print(subprocess.run([sys.executable, str(ROOT / "analysis" / script)],
                         capture_output=True, text=True).stdout[-800:])
display(Markdown("Tables: `analysis/output/tables`  
Figures: `analysis/output/figures`"))


## Notes

Every table in the report is produced by `analysis/make_tables.py`; the report
itself is rendered by `analysis/make_report.py`, which injects these numbers into
`docs/report_fa.template.md`.  No number is typed by hand.